### DESCRIPTION (inspired by Leetcode.com)

Given the root node of a binary tree, write a function to find the number of "good nodes" in the tree. A node X in the tree is considered "good" if in the path from the root to the node X, there are no nodes with a value greater than X's value.

Example

Input:

[4, 2, 7, 1, 3, 6, 9]

Output:

3 # The good nodes are highlighted in green (4, 7, 9)

In [ ]:
class TreeNode:
    def __init__(self, val: int, left: 'TreeNode' = None, right: 'TreeNode' = None):
        self.val = val
        self.left = left
        self.right = right

class Solution_V1:
    def goodNodes(self, root: TreeNode) -> int:
        return self._helper(root, [])

    def _helper(self, root: TreeNode, ancestors: list[int]) -> int:
        if root is None:
            return 0
        else:
            for a in ancestors:
                if a > root.val:
                    return 0 

            ancestors.append(root.val)
            count = 1 + self._helper(root.left, ancestors) + self._helper(root.right, ancestors)
            ancestors.pop()

            return count

### Feedback

Your traversal and backtracking are close, but the early return 0 is the bug. When a node is not good, you should exclude only that node—not stop exploring its descendants. A descendant can still be good relative to all earlier ancestors. For example, in [3,1,4,3,...], node 1 is not good, but its child 3 is good. Replace the early return with a boolean/count check, then recurse regardless:

is_good = all(a <= root.val for a in ancestors)
ancestors.append(root.val)
count = (1 if is_good else 0)
count += self._helper(root.left, ancestors)
count += self._helper(root.right, ancestors)
ancestors.pop()
return count

Your approach is correct in complexity: O(n·h) time and O(h) recursion/list space, though tracking only the maximum ancestor would reduce time to O(n).

In [ ]:
class Solution_V2:
    def goodNodes(self, root: TreeNode) -> int:
        return self._helper(root, -float('inf'))

    def _helper(self, root: TreeNode, max_value: int) -> int:
        if root is None:
            return 0
        else:
            count = 0
            if root.val >= max_value:
                max_value = root.val
                count = 1

            return count + self._helper(root.left, max_value) + self._helper(root.right, max_value)
                        

### Feedback

Your solution is correct and passes all tests. You correctly carry the maximum value along each root-to-node path, count the current node when root.val >= max_value, and pass the updated maximum to both children. It runs in O(n) time and O(h) recursion space, where h is the tree height. 

One interview consideration: with up to 100,000 nodes, a highly skewed tree can exceed Python’s recursion limit and raise RecursionError. An iterative DFS using a stack would avoid that worst-case risk. Otherwise, the -float('inf') initialization safely handles the full value range, including negative values.

In [52]:
class Solution:
    def goodNodes(self, root: TreeNode) -> int:
        count = 0
        stack = [(root, -float('inf'))]

        while stack:
            node, max_value = stack.pop()
            if node is None:
                continue
            
            if node.val >= max_value:
                max_value = node.val
                count += 1
            stack.append((node.left, max_value))
            stack.append((node.right, max_value))
             
        return count

### Feedback

Correct and efficient solution—congratulations. Your iterative DFS avoids Python recursion-depth problems on a tree with up to 100,000 nodes. Carrying max_value in each stack entry correctly preserves the maximum along each root-to-node path, and using >= properly handles equal values. The root is None case also works because it is skipped immediately. 

Time complexity is O(n), with O(h) auxiliary space on average (up to O(n) for a skewed tree). One minor readability improvement would be to push only non-None children, avoiding temporary stack entries, but your current code is already fully correct.

In [4]:
from typing import Callable

class Input:
    def __init__(self, nums: list[int]):
        self.nums = nums
        
class Test:  
    def __init__(self, input: Input, result: int):
        self.input = input
        self.expected_result = result
        self.root = self._createNodefromList(input.nums, 0)

    def _createNodefromList(self, nums: list[int], index: int) -> TreeNode:
        if index >= len(nums) or nums[index] is None:
            return None
        newNode = TreeNode(nums[index])
        newNode.left = self._createNodefromList(nums, 2*index+1)
        newNode.right = self._createNodefromList(nums, 2*index+2)
        return newNode

def run_tests(tests: list[Test], func: Callable[[TreeNode], int]):
    for test in tests:
        result = func(test.root)
        if result == test.expected_result:
            print(f"Test passed for {test.input.nums}")
        else:
            print(f"Test failed for {test.input.nums}. Expected: {test.expected_result}, Actual: {result}")

In [53]:
tests = [
    Test(Input([3]),1),
    Test(Input([3,1]),1),
    Test(Input([3,5]),2),
    Test(Input([3,2,5]),2),
    Test(Input([3,5,7]),3),
    Test(Input([4, 2, 7, 1, 3, 6, 9]),3),
    Test(Input([4, 12, 7, 1, 13, 6, 9]),5),
    Test(Input([3,1,4,3,None,1,5]),4),
    Test(Input([5,1,6,10000]),3)
]

run_tests(tests, Solution().goodNodes)

Test passed for [3]
Test passed for [3, 1]
Test passed for [3, 5]
Test passed for [3, 2, 5]
Test passed for [3, 5, 7]
Test passed for [4, 2, 7, 1, 3, 6, 9]
Test passed for [4, 12, 7, 1, 13, 6, 9]
Test passed for [3, 1, 4, 3, None, 1, 5]
Test passed for [5, 1, 6, 10000]
